In [ ]:
%pip install -q "huggingface-hub>=1.32.0,<2.0"

In [ ]:
from huggingface_hub import hf_hub_download

video_path = hf_hub_download(
    repo_id="atomscott/soccertrack-v2",
    repo_type="dataset",
    filename="mot/clips/117093.mp4",
    local_dir="/content/football_dataset"
)

print("Download successful:", video_path)

In [ ]:
import cv2

video_path = "/content/football_dataset/mot/clips/117093.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Could not open the video")

fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Total frames:", frames)
print("Duration:", round(frames / fps, 2) if fps else "Unknown", "seconds")

cap.release()

In [ ]:
from huggingface_hub import hf_hub_download

annotation_path = hf_hub_download(
    repo_id="atomscott/soccertrack-v2",
    repo_type="dataset",
    filename="mot/117093.txt",
    local_dir="/content/football_dataset"
)

print("Annotations downloaded:", annotation_path)

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolo26n.pt")

results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    classes=[0],
    conf=0.15,
    imgsz=960,
    vid_stride=1,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    project="/content/football_outputs",
    name="baseline",
    exist_ok=True,
    stream=True,
    verbose=False
)

for i, result in enumerate(results):
    if i >= 300:
        break

print("Baseline tracking finished!")

In [ ]:
from pathlib import Path

output_dir = Path("/content/football_outputs/baseline")

print("Generated files:")
for file in output_dir.rglob("*"):
    if file.is_file():
        print(file.name, round(file.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from IPython.display import Video, display
from pathlib import Path

output_dir = Path("/content/football_outputs/baseline")

videos = list(output_dir.glob("*.mp4")) + list(output_dir.glob("*.avi"))

if videos:
    print("Video:", videos[0])
    display(Video(str(videos[0]), embed=True, width=900))
else:
    print("No output video found.")

In [ ]:
import subprocess

input_video = "/content/football_outputs/baseline/117093.avi"
output_video = "/content/football_outputs/baseline/tracked.mp4"

subprocess.run([
    "ffmpeg", "-y",
    "-i", input_video,
    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "28",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    output_video
], check=True)

print("MP4 conversion successful!")

In [ ]:
import subprocess

subprocess.run([
    "ffmpeg", "-y",
    "-i", "/content/football_outputs/baseline/tracked.mp4",
    "-t", "15",
    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "30",
    "-vf", "scale=960:-2",
    "/content/football_outputs/baseline/preview.mp4"
], check=True)

display(Video(
    "/content/football_outputs/baseline/preview.mp4",
    embed=True,
    width=900
))

In [ ]:
from pathlib import Path

annotation_path = Path(
    "/content/football_dataset/mot/117093.txt"
)

if not annotation_path.exists():
    print("Ground-truth file missing!")
else:
    with open(annotation_path, "r") as f:
        for i in range(10):
            line = f.readline()
            if not line:
                break
            print(line.strip())

In [ ]:
import pandas as pd

gt = pd.read_csv(annotation_path, header=None)

print("Dataset shape:", gt.shape)
print("\nFirst 5 rows:")
display(gt.head())

print("\nNumber of columns:", gt.shape[1])

In [ ]:
from ultralytics import YOLO
import cv2
import pandas as pd
from pathlib import Path

video_path = "/content/football_dataset/mot/clips/117093.mp4"
output_dir = Path("/content/football_outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo26n.pt")

cap = cv2.VideoCapture(video_path)
predictions = []

frame_idx = 0
max_frames = 300

while frame_idx < max_frames:
    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        tracker="bytetrack.yaml",
        classes=[0],
        conf=0.15,
        imgsz=960,
        persist=True,
        verbose=False
    )

    boxes = results[0].boxes

    if boxes is not None and boxes.id is not None:
        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confidences = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confidences):
            x1, y1, x2, y2 = box

            predictions.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2 - x1),
                float(y2 - y1),
                float(conf),
                -1, -1, -1
            ])

    frame_idx += 1

cap.release()

columns = [
    "frame", "id", "x", "y",
    "width", "height", "confidence",
    "col8", "col9", "col10"
]

pred_df = pd.DataFrame(predictions, columns=columns)

pred_path = output_dir / "bytetrack_predictions.txt"
pred_df.to_csv(pred_path, index=False, header=False)

print("Frames processed:", frame_idx)
print("Tracking records:", len(pred_df))
print("Unique track IDs:", pred_df["id"].nunique())
print("Saved predictions:", pred_path)

display(pred_df.head())

In [ ]:
%pip install -q motmetrics scipy

In [ ]:
import numpy as np

# Compatibility fix for motmetrics with NumPy 2.x
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

import motmetrics as mm

print("NumPy version:", np.__version__)
print("motmetrics compatibility fix applied!")

In [ ]:
import pandas as pd
import numpy as np
import motmetrics as mm

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None
)

pred = pd.read_csv(
    "/content/football_outputs/evaluation/bytetrack_predictions.txt",
    header=None
)

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

gt.columns = columns
pred.columns = columns

# Evaluate only the same 300 frames
gt = gt[(gt.frame >= 0) & (gt.frame < 300)].copy()
pred = pred[(pred.frame >= 0) & (pred.frame < 300)].copy()

print("Ground-truth records:", len(gt))
print("Prediction records:", len(pred))

acc = mm.MOTAccumulator(auto_id=True)

for frame in range(300):
    gt_frame = gt[gt.frame == frame]
    pred_frame = pred[pred.frame == frame]

    gt_boxes = gt_frame[
        ["x", "y", "w", "h"]
    ].to_numpy(dtype=float)

    pred_boxes = pred_frame[
        ["x", "y", "w", "h"]
    ].to_numpy(dtype=float)

    distances = mm.distances.iou_matrix(
        gt_boxes,
        pred_boxes,
        max_iou=0.5
    )

    acc.update(
        gt_frame["id"].tolist(),
        pred_frame["id"].tolist(),
        distances
    )

mh = mm.metrics.create()

summary = mh.compute(
    acc,
    metrics=[
        "mota",
        "idf1",
        "num_switches",
        "precision",
        "recall",
        "num_false_positives",
        "num_misses"
    ],
    name="YOLO_ByteTrack"
)

display(
    mm.io.render_summary(
        summary,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
)

In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import pandas as pd
import torch

model = YOLO("yolo26n.pt")

video_path = "/content/football_dataset/mot/clips/117093.mp4"
output_dir = Path("/content/football_outputs/evaluation/bytetrack_1536.txt")
output_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(video_path)
predictions = []

frame_idx = 0

while frame_idx < 300:
    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        tracker="bytetrack.yaml",
        classes=[0],
        conf=0.15,
        imgsz=1536,
        device=0 if torch.cuda.is_available() else "cpu",
        persist=True,
        verbose=False
    )

    boxes = results[0].boxes

    if boxes is not None and boxes.id is not None:
        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confs):
            x1, y1, x2, y2 = box

            predictions.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2-x1),
                float(y2-y1),
                float(conf),
                -1, -1, -1
            ])

    frame_idx += 1

cap.release()

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

pred_1280 = pd.DataFrame(predictions, columns=columns)

path_1280 = output_dir / "bytetrack_1280.txt"
pred_1280.to_csv(path_1280, index=False, header=False)

print("Frames processed:", frame_idx)
print("Prediction records:", len(pred_1280))
print("Unique IDs:", pred_1280["id"].nunique())
print("Saved:", path_1280)


In [ ]:

import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

# Compatibility for motmetrics with NumPy 2.x
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

# Load ground truth
gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=columns
)

# All three model predictions
paths = {
    "YOLO_960": "/content/football_outputs/evaluation/bytetrack_predictions.txt",
    "YOLO_1280": "/content/football_outputs/evaluation/bytetrack_1280.txt",
    "YOLO_1536": "/content/football_outputs/evaluation/bytetrack_1536.txt/bytetrack_1536.txt"
}

mh = mm.metrics.create()
summaries = []

gt_subset = gt[
    (gt.frame >= 0) & (gt.frame < 300)
].copy()

gt_groups = dict(tuple(gt_subset.groupby("frame")))

for name, path in paths.items():

    # Verify prediction file exists
    if not Path(path).is_file():
        print(f"File not found for {name}: {path}")
        continue

    pred = pd.read_csv(
        path,
        header=None,
        names=columns
    )

    pred_subset = pred[
        (pred.frame >= 0) & (pred.frame < 300)
    ].copy()

    print(f"\nEvaluating {name}")
    print("Prediction records:", len(pred_subset))
    print("Unique track IDs:", pred_subset["id"].nunique())

    pred_groups = dict(tuple(pred_subset.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(300):

        gt_frame = gt_groups.get(
            frame, gt_subset.iloc[0:0]
        )

        pred_frame = pred_groups.get(
            frame, pred_subset.iloc[0:0]
        )

        distances = mm.distances.iou_matrix(
            gt_frame[["x", "y", "w", "h"]].to_numpy(dtype=float),
            pred_frame[["x", "y", "w", "h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gt_frame["id"].tolist(),
            pred_frame["id"].tolist(),
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "precision",
            "recall",
            "num_false_positives",
            "num_misses"
        ],
        name=name
    )

    summaries.append(summary)

# Display comparison
if summaries:
    comparison = pd.concat(summaries)

    display(
        mm.io.render_summary(
            comparison,
            formatters=mh.formatters,
            namemap=mm.io.motchallenge_metric_names
        )
    )
else:
    print("No prediction files found!")


In [ ]:
model = YOLO("yolo26n.pt")

results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    imgsz=1536,
    conf=0.15,
    classes=[0],
    device=0,
    stream=True,
    verbose=False
)

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

source = Path("/content/football_outputs/evaluation")
destination = Path(
    "/content/drive/MyDrive/Football_CV/baseline_results"
)

destination.mkdir(parents=True, exist_ok=True)

for filename in [
    "bytetrack_predictions.txt",
    "bytetrack_1280.txt",
    "predictions_1536.txt"
]:
    src = source / filename

    if src.is_file():
        shutil.copy2(src, destination / filename)
        print("Saved:", filename)

In [ ]:

import cv2
import torch
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"
OUTPUT_DIR = Path("/content/football_outputs/multisegment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEGMENTS = {
    "B": (2000, 2300),
    "C": (4000, 4300)
}

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

DEVICE = 0 if torch.cuda.is_available() else "cpu"

def run_segment(name, start, end):
    print(f"\nProcessing Segment {name}: {start}-{end-1}")

    # Fresh model and tracker for each segment
    model = YOLO("yolo26n.pt")

    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    predictions = []

    for frame_idx in range(start, end):
        success, frame = cap.read()

        if not success:
            print(f"Video ended at frame {frame_idx}")
            break

        result = model.track(
            frame,
            tracker="bytetrack.yaml",
            classes=[0],
            conf=0.15,
            imgsz=1536,
            device=DEVICE,
            persist=True,
            verbose=False
        )[0]

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confs):
            x1, y1, x2, y2 = box

            predictions.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2 - x1),
                float(y2 - y1),
                float(conf),
                -1, -1, -1
            ])

    cap.release()

    df = pd.DataFrame(predictions, columns=COLUMNS)

    output_path = OUTPUT_DIR / f"segment_{name}_1536.txt"
    df.to_csv(output_path, index=False, header=False)

    print("Tracking records:", len(df))
    print("Unique IDs:", df["id"].nunique())
    print("Saved:", output_path)

    return df

for name, (start, end) in SEGMENTS.items():
    run_segment(name, start, end)

print("\nAll segments processed!")


In [ ]:

import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=COLUMNS
)

experiments = {
    "Segment_A": (
        0, 300,
        "/content/football_outputs/evaluation/bytetrack_1536.txt"
    ),
    "Segment_B": (
        2000, 2300,
        "/content/football_outputs/multisegment/segment_B_1536.txt"
    ),
    "Segment_C": (
        4000, 4300,
        "/content/football_outputs/multisegment/segment_C_1536.txt"
    )
}

mh = mm.metrics.create()
summaries = []

for name, (start, end, path) in experiments.items():

    if not Path(path).is_file():
        print(f"Missing prediction file: {path}")
        continue

    pred = pd.read_csv(path, header=None, names=COLUMNS)

    gf_all = gt[
        (gt.frame >= start) & (gt.frame < end)
    ].copy()

    pf_all = pred[
        (pred.frame >= start) & (pred.frame < end)
    ].copy()

    print(f"\n{name}")
    print("GT records:", len(gf_all))
    print("Prediction records:", len(pf_all))

    gt_groups = dict(tuple(gf_all.groupby("frame")))
    pred_groups = dict(tuple(pf_all.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(start, end):
        gf = gt_groups.get(frame, gf_all.iloc[0:0])
        pf = pred_groups.get(frame, pf_all.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf["id"].tolist(),
            pf["id"].tolist(),
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "precision",
            "recall",
            "num_false_positives",
            "num_misses"
        ],
        name=name
    )

    summaries.append(summary)

comparison = pd.concat(summaries)

display(
    mm.io.render_summary(
        comparison,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
)

# Save actual numeric metrics
result_path = Path(
    "/content/football_outputs/multisegment/segment_metrics.csv"
)
comparison.to_csv(result_path)

print("\nMetrics saved:", result_path)


In [ ]:

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

video_path = "/content/football_dataset/mot/clips/117093.mp4"

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=columns
)

prediction_files = {
    150: "/content/football_outputs/evaluation/bytetrack_1536.txt",
    2150: "/content/football_outputs/multisegment/segment_B_1536.txt",
    4150: "/content/football_outputs/multisegment/segment_C_1536.txt"
}

cap = cv2.VideoCapture(video_path)

fig, axes = plt.subplots(3, 1, figsize=(20, 12))

for ax, (frame_number, pred_path) in zip(
    axes, prediction_files.items()
):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    success, frame = cap.read()

    if not success:
        print("Failed to read frame:", frame_number)
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    predictions = pd.read_csv(
        pred_path, header=None, names=columns
    )

    gt_frame = gt[gt.frame == frame_number]
    pred_frame = predictions[
        predictions.frame == frame_number
    ]

    # Ground truth: green
    for _, row in gt_frame.iterrows():
        x1 = int(row.x)
        y1 = int(row.y)
        x2 = int(row.x + row.w)
        y2 = int(row.y + row.h)

        cv2.rectangle(
            frame, (x1, y1), (x2, y2),
            (0, 255, 0), 3
        )

    # Predictions: red
    for _, row in pred_frame.iterrows():
        x1 = int(row.x)
        y1 = int(row.y)
        x2 = int(row.x + row.w)
        y2 = int(row.y + row.h)

        cv2.rectangle(
            frame, (x1, y1), (x2, y2),
            (255, 0, 0), 2
        )

    ax.imshow(frame)
    ax.set_title(
        f"Frame {frame_number} | "
        f"GT: {len(gt_frame)} | "
        f"Predictions: {len(pred_frame)}"
    )
    ax.axis("off")

cap.release()

plt.tight_layout()
plt.show()


In [ ]:

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"

# Approximate playable pitch polygon
# Coordinates correspond to original 4096x1080 frame
PITCH_POLYGON = np.array([
    [430, 520],
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 4150)

success, frame = cap.read()
cap.release()

if not success:
    raise RuntimeError("Could not read video frame")

overlay = frame.copy()

cv2.polylines(
    overlay,
    [PITCH_POLYGON],
    isClosed=True,
    color=(0, 255, 255),
    thickness=5
)

from google.colab.patches import cv2_imshow

cv2_imshow(cv2.resize(overlay, (1024, 270)))


In [ ]:

import cv2
import numpy as np
import pandas as pd

PITCH_POLYGON = np.array([
    [430, 520],
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=columns
)

prediction_files = {
    "A": "/content/football_outputs/evaluation/bytetrack_1536.txt",
    "B": "/content/football_outputs/multisegment/segment_B_1536.txt",
    "C": "/content/football_outputs/multisegment/segment_C_1536.txt"
}

segments = {
    "A": (0, 300),
    "B": (2000, 2300),
    "C": (4000, 4300)
}

def inside_pitch(df):
    points = zip(
        df["x"] + df["w"] / 2,
        df["y"] + df["h"]
    )

    return np.array([
        cv2.pointPolygonTest(
            PITCH_POLYGON,
            (float(x), float(y)),
            False
        ) >= 0
        for x, y in points
    ], dtype=bool)

for name, (start, end) in segments.items():

    gt_segment = gt[
        (gt.frame >= start) & (gt.frame < end)
    ].copy()

    pred_segment = pd.read_csv(
        prediction_files[name],
        header=None,
        names=columns
    )

    gt_inside = inside_pitch(gt_segment)
    pred_inside = inside_pitch(pred_segment)

    print(f"\nSegment {name}")
    print("GT inside ROI:", gt_inside.sum(), "/", len(gt_segment))
    print("GT outside ROI:", (~gt_inside).sum())
    print("Predictions inside ROI:", pred_inside.sum(), "/", len(pred_segment))
    print("Predictions outside ROI:", (~pred_inside).sum())


In [ ]:

import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

PITCH_POLYGON = np.array([
    [430, 520],
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=columns
)

experiments = {
    "A": (
        0, 300,
        "/content/football_outputs/evaluation/bytetrack_1536.txt"
    ),
    "B": (
        2000, 2300,
        "/content/football_outputs/multisegment/segment_B_1536.txt"
    ),
    "C": (
        4000, 4300,
        "/content/football_outputs/multisegment/segment_C_1536.txt"
    )
}

def inside_pitch(df):
    return np.array([
        cv2.pointPolygonTest(
            PITCH_POLYGON,
            (float(r.x + r.w / 2), float(r.y + r.h)),
            False
        ) >= 0
        for r in df.itertuples()
    ], dtype=bool)

mh = mm.metrics.create()
summaries = []

for segment, (start, end, path) in experiments.items():

    pred = pd.read_csv(path, header=None, names=columns)
    pred = pred[
        (pred.frame >= start) & (pred.frame < end)
    ].copy()

    gt_seg = gt[
        (gt.frame >= start) & (gt.frame < end)
    ].copy()

    filtered = pred[inside_pitch(pred)].copy()

    print(f"\nSegment {segment}")
    print("Original predictions:", len(pred))
    print("Filtered predictions:", len(filtered))

    for label, candidate in [
        ("Original", pred),
        ("Filtered", filtered)
    ]:
        acc = mm.MOTAccumulator(auto_id=True)

        gt_groups = dict(tuple(gt_seg.groupby("frame")))
        pred_groups = dict(tuple(candidate.groupby("frame")))

        for frame in range(start, end):
            gf = gt_groups.get(frame, gt_seg.iloc[0:0])
            pf = pred_groups.get(frame, candidate.iloc[0:0])

            distances = mm.distances.iou_matrix(
                gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
                pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
                max_iou=0.5
            )

            acc.update(
                gf["id"].tolist(),
                pf["id"].tolist(),
                distances
            )

        summary = mh.compute(
            acc,
            metrics=[
                "mota",
                "idf1",
                "num_switches",
                "precision",
                "recall",
                "num_false_positives",
                "num_misses"
            ],
            name=f"{segment}_{label}"
        )

        summaries.append(summary)

comparison = pd.concat(summaries)

display(mm.io.render_summary(
    comparison,
    formatters=mh.formatters,
    namemap=mm.io.motchallenge_metric_names
))

output_dir = Path("/content/football_outputs/roi")
output_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output_dir / "roi_comparison.csv")


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

video_path = "/content/football_dataset/mot/clips/117093.mp4"

# Reuse gt, PITCH_POLYGON and inside_pitch from previous cells
gt_a = gt[(gt.frame >= 0) & (gt.frame < 300)].copy()

outside_gt = gt_a[~inside_pitch(gt_a)].copy()

print("Excluded GT records:", len(outside_gt))
print("Affected frames:", outside_gt.frame.nunique())
print("Affected track IDs:", outside_gt.id.unique())

# Show an affected frame, if any
if not outside_gt.empty:
    frame_number = int(outside_gt.frame.iloc[0])

    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    success, frame = cap.read()
    cap.release()

    if not success:
        raise RuntimeError("Failed to read selected frame")

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    cv2.polylines(
        frame, [PITCH_POLYGON],
        True, (255, 255, 0), 4
    )

    selected = outside_gt[
        outside_gt.frame == frame_number
    ]

    for row in selected.itertuples():
        p1 = (int(row.x), int(row.y))
        p2 = (
            int(row.x + row.w),
            int(row.y + row.h)
        )

        cv2.rectangle(
            frame, p1, p2,
            (255, 0, 0), 5
        )

        cv2.putText(
            frame, f"GT {row.id}",
            (p1[0], max(20, p1[1] - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (255, 0, 0), 2
        )

    plt.figure(figsize=(20, 6))
    plt.imshow(frame)
    plt.title(f"Excluded ground-truth boxes — Frame {frame_number}")
    plt.axis("off")
    plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Reuse gt_a, outside_gt, and PITCH_POLYGON
cap = cv2.VideoCapture(
    "/content/football_dataset/mot/clips/117093.mp4"
)
cap.set(cv2.CAP_PROP_POS_FRAMES, 130)
success, frame = cap.read()
cap.release()

if not success:
    raise RuntimeError("Unable to read frame 130")

frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Draw current polygon
cv2.polylines(
    frame, [PITCH_POLYGON],
    True, (255, 255, 0), 4
)

# Draw all excluded GT footpoints from Segment A
for row in outside_gt.itertuples():
    foot_x = int(row.x + row.w / 2)
    foot_y = int(row.y + row.h)

    cv2.circle(
        frame,
        (foot_x, foot_y),
        8,
        (255, 0, 0),
        -1
    )

plt.figure(figsize=(22, 7))
plt.imshow(frame)
plt.title("Excluded GT footpoints — Segment A (all 300 frames)")
plt.axis("off")
plt.show()

print("Excluded GT records:", len(outside_gt))
print("Excluded IDs:", sorted(outside_gt.id.unique()))

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

PITCH_POLYGON_V2 = np.array([
    [150, 570],     # Extended lower-left boundary
    [350, 440],     # Left-side transition
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

cap = cv2.VideoCapture(
    "/content/football_dataset/mot/clips/117093.mp4"
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 130)
success, frame = cap.read()
cap.release()

if not success:
    raise RuntimeError("Unable to read frame 130")

frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

cv2.polylines(
    frame,
    [PITCH_POLYGON_V2],
    True,
    (255, 255, 0),
    5
)

plt.figure(figsize=(20, 6))
plt.imshow(frame)
plt.axis("off")
plt.title("Revised Pitch ROI — Version 2")
plt.show()

In [ ]:

import cv2
import numpy as np
import pandas as pd

# Revised pitch polygon (Version 2)
PITCH_POLYGON_V2 = np.array([
    [150, 570],
    [350, 440],
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

# Define ROI coverage function
def roi_coverage(df, polygon):
    inside = []

    for row in df.itertuples(index=False):
        foot_x = float(row.x + row.w / 2)
        foot_y = float(row.y + row.h)

        valid = cv2.pointPolygonTest(
            polygon,
            (foot_x, foot_y),
            False
        ) >= 0

        inside.append(valid)

    return np.array(inside, dtype=bool)

# Column definitions
columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

# Ground truth
gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=columns
)

# Experiment paths
experiments = {
    "A": (
        0, 300,
        "/content/football_outputs/evaluation/bytetrack_1536.txt"
    ),
    "B": (
        2000, 2300,
        "/content/football_outputs/multisegment/segment_B_1536.txt"
    ),
    "C": (
        4000, 4300,
        "/content/football_outputs/multisegment/segment_C_1536.txt"
    )
}

for name, (start, end, path) in experiments.items():

    gt_seg = gt[
        (gt.frame >= start) & (gt.frame < end)
    ].copy()

    pred = pd.read_csv(
        path, header=None, names=columns
    )

    pred = pred[
        (pred.frame >= start) &
        (pred.frame < end)
    ].copy()

    gt_mask = roi_coverage(gt_seg, PITCH_POLYGON_V2)
    pred_mask = roi_coverage(pred, PITCH_POLYGON_V2)

    print(f"\nSegment {name}")

    print(
        f"GT retained: {gt_mask.sum()} / {len(gt_seg)}"
    )
    print("GT excluded:", (~gt_mask).sum())

    print(
        f"Predictions retained: {pred_mask.sum()} / {len(pred)}"
    )
    print("Predictions excluded:", (~pred_mask).sum())


In [ ]:

import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

mh = mm.metrics.create()
summaries = []

output_dir = Path("/content/football_outputs/roi_v2")
output_dir.mkdir(parents=True, exist_ok=True)

for segment, (start, end, path) in experiments.items():

    gt_seg = gt[
        (gt.frame >= start) & (gt.frame < end)
    ].copy()

    pred = pd.read_csv(
        path, header=None, names=columns
    )

    pred = pred[
        (pred.frame >= start) & (pred.frame < end)
    ].copy()

    # Filter predictions only, not ground truth
    filtered = pred[
        roi_coverage(pred, PITCH_POLYGON_V2)
    ].copy()

    filtered.to_csv(
        output_dir / f"segment_{segment}_filtered.txt",
        index=False,
        header=False
    )

    gt_groups = dict(tuple(gt_seg.groupby("frame")))

    for label, candidate in [
        ("Original", pred),
        ("ROI_V2", filtered)
    ]:

        pred_groups = dict(tuple(candidate.groupby("frame")))
        acc = mm.MOTAccumulator(auto_id=True)

        for frame in range(start, end):
            gf = gt_groups.get(frame, gt_seg.iloc[0:0])
            pf = pred_groups.get(frame, candidate.iloc[0:0])

            distances = mm.distances.iou_matrix(
                gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
                pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
                max_iou=0.5
            )

            acc.update(
                gf["id"].tolist(),
                pf["id"].tolist(),
                distances
            )

        summary = mh.compute(
            acc,
            metrics=[
                "mota", "idf1", "num_switches",
                "precision", "recall",
                "num_false_positives", "num_misses"
            ],
            name=f"{segment}_{label}"
        )

        summaries.append(summary)

comparison_v2 = pd.concat(summaries)

display(
    mm.io.render_summary(
        comparison_v2,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
)

comparison_v2.to_csv(output_dir / "roi_v2_metrics.csv")

print("Results saved:", output_dir)


In [ ]:
from pathlib import Path
import shutil

source = Path("/content/football_outputs/roi_v2")
destination = Path(
    "/content/drive/MyDrive/Football_CV/roi_v2_results"
)

destination.mkdir(parents=True, exist_ok=True)

for file in source.iterdir():
    if file.is_file():
        shutil.copy2(file, destination / file.name)

print("ROI V2 results backed up!")

In [ ]:

from huggingface_hub import hf_hub_download

repo = "atomscott/soccertrack-v2"

training_files = [
    "mot/clips/117092.mp4",
    "mot/117092.txt"
]

for filename in training_files:
    path = hf_hub_download(
        repo_id=repo,
        repo_type="dataset",
        filename=filename,
        local_dir="/content/football_dataset"
    )
    print("Downloaded:", path)

print("\nTraining files ready!")


In [ ]:

import cv2
import pandas as pd

video_path = "/content/football_dataset/mot/clips/117092.mp4"
annotation_path = "/content/football_dataset/mot/117092.txt"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Training video could not be opened")

print("Resolution:",
      int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
      "x",
      int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

cap.release()

gt_train = pd.read_csv(annotation_path, header=None)

print("Annotation records:", len(gt_train))
print("Unique IDs:", gt_train[1].nunique())
print("Columns:", gt_train.shape[1])
print("Frame range:", gt_train[0].min(), "-", gt_train[0].max())

display(gt_train.head())


In [ ]:

import cv2
import pandas as pd
from pathlib import Path

DATASET_ROOT = Path("/content/football_yolo")

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

SPLITS = {
    "train": {
        "video": "/content/football_dataset/mot/clips/117092.mp4",
        "labels": "/content/football_dataset/mot/117092.txt",
        "stride": 15,
        "max_images": 400
    },
    "val": {
        "video": "/content/football_dataset/mot/clips/117093.mp4",
        "labels": "/content/football_dataset/mot/117093.txt",
        "stride": 20,
        "max_images": 300
    }
}

def xywh_to_yolo(row, image_w, image_h):
    x1 = max(0.0, float(row.x))
    y1 = max(0.0, float(row.y))
    x2 = min(float(image_w), float(row.x + row.w))
    y2 = min(float(image_h), float(row.y + row.h))

    if x2 <= x1 or y2 <= y1:
        return None

    xc = ((x1 + x2) / 2) / image_w
    yc = ((y1 + y2) / 2) / image_h
    w = (x2 - x1) / image_w
    h = (y2 - y1) / image_h

    return f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"

for split, config in SPLITS.items():

    image_dir = DATASET_ROOT / "images" / split
    label_dir = DATASET_ROOT / "labels" / split

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    annotations = pd.read_csv(
        config["labels"],
        header=None,
        names=COLUMNS
    )

    grouped = dict(tuple(annotations.groupby("frame")))

    cap = cv2.VideoCapture(config["video"])

    if not cap.isOpened():
        raise RuntimeError("Could not open " + config["video"])

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved = 0

    for frame_idx in range(0, total_frames, config["stride"]):

        if saved >= config["max_images"]:
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            continue

        h, w = frame.shape[:2]

        image_name = f"{split}_{frame_idx:06d}"

        image_path = image_dir / f"{image_name}.jpg"
        label_path = label_dir / f"{image_name}.txt"

        cv2.imwrite(
            str(image_path),
            frame,
            [cv2.IMWRITE_JPEG_QUALITY, 90]
        )

        rows = grouped.get(frame_idx)
        labels = []

        if rows is not None:
            for row in rows.itertuples(index=False):
                label = xywh_to_yolo(row, w, h)
                if label is not None:
                    labels.append(label)

        label_path.write_text("\n".join(labels))

        saved += 1

    cap.release()

    print(f"{split}: {saved} images prepared")

print("Dataset preparation complete!")


In [ ]:

from pathlib import Path
import yaml

dataset_config = {
    "path": "/content/football_yolo",
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": {0: "person"}
}

config_path = Path("/content/football_yolo/data.yaml")

with open(config_path, "w") as f:
    yaml.safe_dump(dataset_config, f, sort_keys=False)

print(config_path.read_text())


In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

image_dir = Path("/content/football_yolo/images/train")
label_dir = Path("/content/football_yolo/labels/train")

image_path = sorted(image_dir.glob("*.jpg"))[10]
label_path = label_dir / (image_path.stem + ".txt")

image = cv2.imread(str(image_path))
h, w = image.shape[:2]

for line in label_path.read_text().splitlines():
    cls, xc, yc, bw, bh = map(float, line.split())

    x1 = int((xc - bw / 2) * w)
    y1 = int((yc - bh / 2) * h)
    x2 = int((xc + bw / 2) * w)
    y2 = int((yc + bh / 2) * h)

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        3
    )

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18, 7))
plt.imshow(image)
plt.title("YOLO Training Annotations")
plt.axis("off")
plt.show()


In [ ]:

from ultralytics import YOLO
import torch

assert torch.cuda.is_available(), "Enable GPU in Colab first"

model = YOLO("yolo26n.pt")

results = model.train(
    data="/content/football_yolo/data.yaml",
    epochs=30,
    imgsz=1280,
    batch=2,
    optimizer="AdamW",
    lr0=0.001,
    patience=10,
    device=0,
    workers=2,
    cache=False,
    project="/content/football_outputs/training",
    name="yolo26n_football_v1",
    exist_ok=True,
    plots=True
)

print("Training complete!")


In [ ]:

from ultralytics import YOLO

model = YOLO(
    "/content/football_outputs/training/yolo26n_football_v1/weights/best.pt"
)

metrics = model.val(
    data="/content/football_yolo/data.yaml",
    imgsz=1280,
    batch=2,
    device=0
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)


In [ ]:

from ultralytics import YOLO
import pandas as pd

models = {
    "Pretrained": "yolo26n.pt",
    "Fine_Tuned_V1": (
        "/content/football_outputs/training/"
        "yolo26n_football_v1/weights/best.pt"
    )
}

results = []

for name, path in models.items():

    print(f"\nEvaluating {name}")

    model = YOLO(path)

    metrics = model.val(
        data="/content/football_yolo/data.yaml",
        imgsz=1280,
        batch=2,
        device=0,
        conf=0.001,
        iou=0.7,
        plots=False,
        verbose=False
    )

    results.append({
        "Model": name,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map
    })

comparison_df = pd.DataFrame(results)

display(comparison_df.round(4))


In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)

destination = Path(
    "/content/drive/MyDrive/Football_CV/models/"
    "yolo26n_football_v1_best.pt"
)

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

print("Saved:", destination)

In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import pandas as pd
import torch

VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"
MODEL_PATH = (
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)

OUTPUT_DIR = Path(
    "/content/football_outputs/finetuned_tracking"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEGMENTS = {
    "A": (0, 300),
    "B": (2000, 2300),
    "C": (4000, 4300)
}

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

device = 0 if torch.cuda.is_available() else "cpu"

for name, (start, end) in SEGMENTS.items():

    print(f"\nTracking Segment {name}")

    # Fresh tracker for each segment
    model = YOLO(MODEL_PATH)

    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    rows = []

    for frame_idx in range(start, end):

        success, frame = cap.read()

        if not success:
            break

        result = model.track(
            frame,
            tracker="bytetrack.yaml",
            classes=[0],
            conf=0.15,
            imgsz=1536,
            device=device,
            persist=True,
            verbose=False
        )[0]

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confs):

            x1, y1, x2, y2 = box

            rows.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2 - x1),
                float(y2 - y1),
                float(conf),
                -1, -1, -1
            ])

    cap.release()

    df = pd.DataFrame(rows, columns=COLUMNS)

    output_path = OUTPUT_DIR / f"segment_{name}_finetuned.txt"
    df.to_csv(output_path, index=False, header=False)

    print("Predictions:", len(df))
    print("Unique IDs:", df["id"].nunique())
    print("Saved:", output_path)

print("\nFine-tuned tracking complete!")


In [ ]:

import numpy as np
import pandas as pd
import motmetrics as mm

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=COLUMNS
)

mh = mm.metrics.create()
summaries = []

for name, (start, end) in SEGMENTS.items():

    path = OUTPUT_DIR / f"segment_{name}_finetuned.txt"
    pred = pd.read_csv(path, header=None, names=COLUMNS)

    # Reuse ROI V2 function and polygon from previous cells
    filtered = pred[
        roi_coverage(pred, PITCH_POLYGON_V2)
    ].copy()

    gt_seg = gt[
        (gt.frame >= start) & (gt.frame < end)
    ]

    gt_groups = dict(tuple(gt_seg.groupby("frame")))
    pred_groups = dict(tuple(filtered.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(start, end):

        gf = gt_groups.get(frame, gt_seg.iloc[0:0])
        pf = pred_groups.get(frame, filtered.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf["id"].tolist(),
            pf["id"].tolist(),
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota", "idf1", "num_switches",
            "precision", "recall",
            "num_false_positives", "num_misses"
        ],
        name=f"{name}_FineTuned_ROI_V2"
    )

    summaries.append(summary)

comparison = pd.concat(summaries)

display(
    mm.io.render_summary(
        comparison,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
)

comparison.to_csv(
    OUTPUT_DIR / "finetuned_tracking_metrics.csv"
)


In [ ]:

from ultralytics import YOLO
import cv2
import pandas as pd
import torch
from pathlib import Path

VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"
MODEL_PATH = (
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)

OUTPUT_DIR = Path("/content/football_outputs/conf_experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = [0.15, 0.25, 0.35]
START, END = 2000, 2300

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

device = 0 if torch.cuda.is_available() else "cpu"

for confidence in THRESHOLDS:

    print(f"\nTesting confidence = {confidence}")

    # Fresh tracker for each run
    model = YOLO(MODEL_PATH)

    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, START)

    rows = []

    for frame_idx in range(START, END):
        success, frame = cap.read()

        if not success:
            break

        result = model.track(
            frame,
            tracker="bytetrack.yaml",
            classes=[0],
            conf=confidence,
            imgsz=1536,
            device=device,
            persist=True,
            verbose=False
        )[0]

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confs):
            x1, y1, x2, y2 = box

            rows.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2 - x1),
                float(y2 - y1),
                float(conf),
                -1, -1, -1
            ])

    cap.release()

    df = pd.DataFrame(rows, columns=COLUMNS)

    filename = f"segment_B_conf_{confidence:.2f}.txt"
    output_path = OUTPUT_DIR / filename

    df.to_csv(output_path, index=False, header=False)

    print("Predictions:", len(df))
    print("Unique IDs:", df.id.nunique())
    print("Saved:", output_path)

print("\nAll confidence experiments completed!")


In [ ]:

import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

# NumPy 2.x compatibility
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

PITCH_POLYGON_V2 = np.array([
    [150, 570],
    [350, 440],
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

def roi_coverage(df, polygon):
    return np.array([
        cv2.pointPolygonTest(
            polygon,
            (float(row.x + row.w / 2),
             float(row.y + row.h)),
            False
        ) >= 0
        for row in df.itertuples(index=False)
    ], dtype=bool)

GT_PATH = "/content/football_dataset/mot/117093.txt"
OUTPUT_DIR = Path("/content/football_outputs/conf_experiments")

gt = pd.read_csv(
    GT_PATH, header=None, names=COLUMNS
)

START, END = 2000, 2300

gt_seg = gt[
    (gt.frame >= START) & (gt.frame < END)
].copy()

gt_groups = dict(tuple(gt_seg.groupby("frame")))

mh = mm.metrics.create()
summaries = []

for confidence in [0.15, 0.25, 0.35]:

    path = OUTPUT_DIR / f"segment_B_conf_{confidence:.2f}.txt"

    pred = pd.read_csv(
        path, header=None, names=COLUMNS
    )

    pred = pred[
        (pred.frame >= START) & (pred.frame < END)
    ].copy()

    filtered = pred[
        roi_coverage(pred, PITCH_POLYGON_V2)
    ].copy()

    print(f"\nConfidence: {confidence}")
    print("Original records:", len(pred))
    print("After ROI:", len(filtered))

    pred_groups = dict(tuple(filtered.groupby("frame")))
    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(START, END):

        gf = gt_groups.get(frame, gt_seg.iloc[0:0])
        pf = pred_groups.get(frame, filtered.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf["id"].tolist(),
            pf["id"].tolist(),
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "precision",
            "recall",
            "num_false_positives",
            "num_misses"
        ],
        name=f"Conf_{confidence:.2f}"
    )

    summaries.append(summary)

comparison = pd.concat(summaries)

display(
    mm.io.render_summary(
        comparison,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
)

comparison.to_csv(
    OUTPUT_DIR / "confidence_comparison.csv"
)

print("\nResults saved successfully!")


In [ ]:
import pandas as pd

path = "/content/football_outputs/conf_experiments/segment_B_conf_0.35.txt"

df = pd.read_csv(path, header=None, names=COLUMNS)

print("X range:", df["x"].min(), df["x"].max())
print("Y range:", df["y"].min(), df["y"].max())

print("Maximum X2:", (df["x"] + df["w"]).max())
print("Maximum Y2:", (df["y"] + df["h"]).max())

In [ ]:
import pandas as pd
import numpy as np
import cv2

files = {
    "A": "/content/football_outputs/finetuned_tracking/segment_A_finetuned.txt",
    "B": "/content/football_outputs/finetuned_tracking/segment_B_finetuned.txt",
    "C": "/content/football_outputs/finetuned_tracking/segment_C_finetuned.txt"
}

for name, path in files.items():
    df = pd.read_csv(path, header=None, names=COLUMNS)

    inside = roi_coverage(df, PITCH_POLYGON_V2)

    print(f"\nSegment {name}")
    print("Total detections:", len(df))
    print("Inside ROI:", inside.sum())
    print("Outside ROI:", (~inside).sum())
    print("X range:", round(df.x.min(), 1), "-", round((df.x + df.w).max(), 1))
    print("Y range:", round(df.y.min(), 1), "-", round((df.y + df.h).max(), 1))

In [ ]:

from ultralytics import YOLO
import cv2
import pandas as pd
import torch
from pathlib import Path

VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"

MODEL_PATH = (
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)

OUTPUT_DIR = Path("/content/football_outputs/conf_experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEGMENTS = {
    "A": (0, 300),
    "C": (4000, 4300)
}

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

device = 0 if torch.cuda.is_available() else "cpu"

for name, (start, end) in SEGMENTS.items():

    print(f"\nProcessing Segment {name}")

    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(VIDEO_PATH)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    rows = []

    for frame_idx in range(start, end):

        success, frame = cap.read()

        if not success:
            break

        result = model.track(
            frame,
            tracker="bytetrack.yaml",
            classes=[0],
            conf=0.35,
            imgsz=1536,
            device=device,
            persist=True,
            verbose=False
        )[0]

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confs):
            x1, y1, x2, y2 = box

            rows.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2 - x1),
                float(y2 - y1),
                float(conf),
                -1, -1, -1
            ])

    cap.release()

    df = pd.DataFrame(rows, columns=COLUMNS)

    output_path = OUTPUT_DIR / f"segment_{name}_conf_0.35.txt"

    df.to_csv(output_path, index=False, header=False)

    print("Frames:", end - start)
    print("Detections:", len(df))
    print("Unique IDs:", df["id"].nunique())
    print("Saved:", output_path)

print("\nTracking complete!")


In [ ]:

import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

PITCH_POLYGON_V2 = np.array([
    [150, 570],
    [350, 440],
    [700, 330],
    [1250, 175],
    [1900, 115],
    [2700, 130],
    [3400, 230],
    [3950, 550],
    [3550, 720],
    [2100, 720],
    [700, 680]
], dtype=np.int32)

def roi_coverage(df, polygon):
    return np.array([
        cv2.pointPolygonTest(
            polygon,
            (
                float(row.x + row.w / 2),
                float(row.y + row.h)
            ),
            False
        ) >= 0
        for row in df.itertuples(index=False)
    ], dtype=bool)

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=COLUMNS
)

segments = {
    "A": (0, 300),
    "B": (2000, 2300),
    "C": (4000, 4300)
}

output_dir = Path("/content/football_outputs/conf_experiments")

mh = mm.metrics.create()
summaries = []

for name, (start, end) in segments.items():

    path = output_dir / f"segment_{name}_conf_0.35.txt"

    if not path.is_file():
        print("Missing:", path)
        continue

    pred = pd.read_csv(path, header=None, names=COLUMNS)
    pred = pred[
        (pred.frame >= start) & (pred.frame < end)
    ]

    filtered = pred[
        roi_coverage(pred, PITCH_POLYGON_V2)
    ].copy()

    gt_seg = gt[
        (gt.frame >= start) & (gt.frame < end)
    ]

    gt_groups = dict(tuple(gt_seg.groupby("frame")))
    pred_groups = dict(tuple(filtered.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(start, end):
        gf = gt_groups.get(frame, gt_seg.iloc[0:0])
        pf = pred_groups.get(frame, filtered.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf["id"].tolist(),
            pf["id"].tolist(),
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "precision",
            "recall",
            "num_false_positives",
            "num_misses"
        ],
        name=f"Segment_{name}_conf035"
    )

    summaries.append(summary)

if summaries:
    comparison = pd.concat(summaries)

    display(mm.io.render_summary(
        comparison,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    ))

    comparison.to_csv(
        output_dir / "all_segments_conf035.csv"
    )


In [ ]:

from pathlib import Path
import ultralytics
import yaml

config_path = (
    Path(ultralytics.__file__).parent
    / "cfg" / "trackers" / "bytetrack.yaml"
)

with open(config_path) as f:
    tracker_config = yaml.safe_load(f)

print("Current ByteTrack configuration:\n")

for key, value in tracker_config.items():
    print(f"{key}: {value}")


In [ ]:

from pathlib import Path
import yaml

config_dir = Path(
    "/content/football_outputs/tracker_configs"
)
config_dir.mkdir(parents=True, exist_ok=True)

experiments = {
    "baseline": {},
    "longer_buffer": {
        "track_buffer": 60
    },
    "higher_match_thresh": {
        "match_thresh": 0.9
    },
    "higher_new_track_thresh": {
        "new_track_thresh": 0.45
    }
}

for name, changes in experiments.items():

    config = tracker_config.copy()
    config.update(changes)

    output_path = config_dir / f"{name}.yaml"

    with open(output_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print(f"{name}: {output_path}")
    print(changes if changes else "Original settings")
    print()


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import pandas as pd
import torch

VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"
MODEL_PATH = (
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)

CONFIG_DIR = Path("/content/football_outputs/tracker_configs")
OUTPUT_DIR = Path("/content/football_outputs/tracker_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = [
    "longer_buffer",
    "higher_match_thresh",
    "higher_new_track_thresh"
]

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

START, END = 2000, 2300
device = 0 if torch.cuda.is_available() else "cpu"

for name in EXPERIMENTS:

    print(f"\nRunning: {name}")

    model = YOLO(MODEL_PATH)
    tracker_path = str(CONFIG_DIR / f"{name}.yaml")

    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, START)

    rows = []

    for frame_idx in range(START, END):

        success, frame = cap.read()
        if not success:
            break

        result = model.track(
            frame,
            tracker=tracker_path,
            classes=[0],
            conf=0.35,
            imgsz=1536,
            device=device,
            persist=True,
            verbose=False
        )[0]

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(xyxy, ids, confs):
            x1, y1, x2, y2 = box

            rows.append([
                frame_idx,
                int(track_id),
                float(x1),
                float(y1),
                float(x2-x1),
                float(y2-y1),
                float(conf),
                -1, -1, -1
            ])

    cap.release()

    df = pd.DataFrame(rows, columns=COLUMNS)

    output_path = OUTPUT_DIR / f"{name}.txt"
    df.to_csv(output_path, index=False, header=False)

    print("Predictions:", len(df))
    print("Unique IDs:", df["id"].nunique())
    print("Saved:", output_path)

print("\nTracker experiments completed!")


In [ ]:

import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

PITCH_POLYGON_V2 = np.array([
    [150,570], [350,440], [700,330],
    [1250,175], [1900,115], [2700,130],
    [3400,230], [3950,550], [3550,720],
    [2100,720], [700,680]
], dtype=np.int32)

def roi_coverage(df, polygon):
    return np.array([
        cv2.pointPolygonTest(
            polygon,
            (float(r.x + r.w/2), float(r.y + r.h)),
            False
        ) >= 0
        for r in df.itertuples(index=False)
    ], dtype=bool)

GT_PATH = "/content/football_dataset/mot/117093.txt"

gt = pd.read_csv(
    GT_PATH, header=None, names=COLUMNS
)

START, END = 2000, 2300

gt_seg = gt[
    (gt.frame >= START) & (gt.frame < END)
]
gt_groups = dict(tuple(gt_seg.groupby("frame")))

EXPERIMENT_PATHS = {
    "baseline": (
        "/content/football_outputs/conf_experiments/"
        "segment_B_conf_0.35.txt"
    ),
    "longer_buffer": (
        "/content/football_outputs/tracker_tuning/"
        "longer_buffer.txt"
    ),
    "higher_match_thresh": (
        "/content/football_outputs/tracker_tuning/"
        "higher_match_thresh.txt"
    ),
    "higher_new_track_thresh": (
        "/content/football_outputs/tracker_tuning/"
        "higher_new_track_thresh.txt"
    )
}

mh = mm.metrics.create()
summaries = []

for name, path in EXPERIMENT_PATHS.items():

    pred = pd.read_csv(path, header=None, names=COLUMNS)
    pred = pred[
        (pred.frame >= START) & (pred.frame < END)
    ].copy()

    filtered = pred[
        roi_coverage(pred, PITCH_POLYGON_V2)
    ].copy()

    pred_groups = dict(tuple(filtered.groupby("frame")))
    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(START, END):

        gf = gt_groups.get(frame, gt_seg.iloc[0:0])
        pf = pred_groups.get(frame, filtered.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            pf[["x", "y", "w", "h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf["id"].tolist(),
            pf["id"].tolist(),
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "precision",
            "recall",
            "num_false_positives",
            "num_misses"
        ],
        name=name
    )

    summaries.append(summary)

comparison = pd.concat(summaries)

display(
    mm.io.render_summary(
        comparison,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
)

output_path = (
    "/content/football_outputs/tracker_tuning/"
    "tracker_comparison.csv"
)

comparison.to_csv(output_path)
print("Saved:", output_path)


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import pandas as pd
import torch

MODEL_PATH = (
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)
VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"

TRACKER_PATH = (
    "/content/football_outputs/tracker_configs/"
    "higher_new_track_thresh.yaml"
)
OUTPUT_DIR = Path("/content/football_outputs/tracker_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

SEGMENTS = {
    "A": (0, 300),
    "C": (4000, 4300)
}

assert Path(TRACKER_PATH).exists()
assert Path(MODEL_PATH).exists()

for name, (start, end) in SEGMENTS.items():
    print(f"\nProcessing Segment {name}")

    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    rows = []
    frames_read = 0

    for frame_idx in range(start, end):
        ok, frame = cap.read()
        if not ok:
            break

        frames_read += 1

        result = model.track(
            frame,
            tracker=TRACKER_PATH,
            classes=[0],
            conf=0.35,
            imgsz=1536,
            device=0 if torch.cuda.is_available() else "cpu",
            persist=True,
            verbose=False
        )[0]

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        for box, track_id, conf in zip(
            boxes.xyxy.cpu().numpy(),
            boxes.id.int().cpu().numpy(),
            boxes.conf.cpu().numpy()
        ):
            x1, y1, x2, y2 = map(float, box)

            rows.append([
                frame_idx, int(track_id),
                x1, y1, x2-x1, y2-y1,
                float(conf), -1, -1, -1
            ])

    cap.release()

    assert frames_read == end - start, (
        f"Only read {frames_read} frames for {name}"
    )

    df = pd.DataFrame(rows, columns=COLUMNS)

    path = OUTPUT_DIR / f"segment_{name}_newtrack045.txt"
    df.to_csv(path, header=False, index=False)

    print("Tracking records:", len(df))
    print("Unique IDs:", df.id.nunique())
    print("Saved:", path)

print("\nAll segments ready for evaluation.")


In [ ]:

import numpy as np
import pandas as pd
import cv2
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

ROI = np.array([
    [150,570], [350,440], [700,330],
    [1250,175], [1900,115], [2700,130],
    [3400,230], [3950,550], [3550,720],
    [2100,720], [700,680]
], dtype=np.int32)

ROOT = Path("/content/football_outputs/tracker_tuning")

SEGMENTS = {
    "A": (0, 300, ROOT / "segment_A_newtrack045.txt"),
    "B": (2000, 2300, ROOT / "higher_new_track_thresh.txt"),
    "C": (4000, 4300, ROOT / "segment_C_newtrack045.txt")
}

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None,
    names=COLUMNS
)

mh = mm.metrics.create()
summaries = []

for name, (start, end, path) in SEGMENTS.items():
    assert path.exists(), f"Missing {path}"

    pred = pd.read_csv(path, header=None, names=COLUMNS)

    mask = np.array([
        cv2.pointPolygonTest(
            ROI, (float(r.x + r.w/2), float(r.y + r.h)),
            False
        ) >= 0
        for r in pred.itertuples(index=False)
    ], dtype=bool)

    pred = pred.loc[mask].copy()
    gt_seg = gt[(gt.frame >= start) & (gt.frame < end)]

    gt_groups = dict(tuple(gt_seg.groupby("frame")))
    pred_groups = dict(tuple(pred.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame_idx in range(start, end):
        gf = gt_groups.get(frame_idx, gt_seg.iloc[0:0])
        pf = pred_groups.get(frame_idx, pred.iloc[0:0])

        d = mm.distances.iou_matrix(
            gf[["x","y","w","h"]].to_numpy(dtype=float),
            pf[["x","y","w","h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf.id.tolist(),
            pf.id.tolist(),
            d
        )

    summary = mh.compute(
        acc,
        metrics=[
            "mota", "idf1", "num_switches",
            "precision", "recall",
            "num_false_positives", "num_misses"
        ],
        name=f"Segment_{name}_NewTrack045"
    )
    summaries.append(summary)

comparison = pd.concat(summaries)

display(mm.io.render_summary(
    comparison,
    formatters=mh.formatters,
    namemap=mm.io.motchallenge_metric_names
))

comparison.to_csv(ROOT / "all_segments_newtrack045.csv")
print("Saved all_segments_newtrack045.csv")


In [ ]:

from pathlib import Path
from ultralytics import YOLO
import cv2
import yaml
import pandas as pd

ROOT = Path("/content/football_outputs/tracker_tuning")
ROOT.mkdir(parents=True, exist_ok=True)

config = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.25,
    "track_low_thresh": 0.10,
    "new_track_thresh": 0.45,
    "track_buffer": 30,
    "match_thresh": 0.90,
    "fuse_score": True
}

tracker_path = ROOT / "combined_045_090.yaml"
with open(tracker_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

model = YOLO(
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)

video_path = "/content/football_dataset/mot/clips/117093.mp4"
cap = cv2.VideoCapture(video_path)

START, END = 2000, 2300
cap.set(cv2.CAP_PROP_POS_FRAMES, START)

rows = []
processed = 0

for frame_idx in range(START, END):
    success, frame = cap.read()
    if not success:
        break

    processed += 1

    result = model.track(
        frame,
        tracker=str(tracker_path),
        conf=0.35,
        imgsz=1536,
        classes=[0],
        device=0,
        persist=True,
        verbose=False
    )[0]

    boxes = result.boxes

    if boxes is None or boxes.id is None:
        continue

    for box, track_id, confidence in zip(
        boxes.xyxy.cpu().numpy(),
        boxes.id.int().cpu().numpy(),
        boxes.conf.cpu().numpy()
    ):
        x1, y1, x2, y2 = map(float, box)

        rows.append([
            frame_idx, int(track_id),
            x1, y1, x2-x1, y2-y1,
            float(confidence), -1, -1, -1
        ])

cap.release()

assert processed == END - START, (
    f"Expected 300 frames, processed {processed}"
)

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

predictions = pd.DataFrame(rows, columns=columns)

output_path = ROOT / "combined_045_090.txt"
predictions.to_csv(output_path, index=False, header=False)

print("Frames processed:", processed)
print("Tracking records:", len(predictions))
print("Unique tracking IDs:", predictions.id.nunique())
print("Saved:", output_path)


In [ ]:

import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

ROOT = Path("/content/football_outputs/tracker_tuning")
COLUMNS = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

ROI = np.array([
    [150,570], [350,440], [700,330],
    [1250,175], [1900,115], [2700,130],
    [3400,230], [3950,550],
    [3550,720], [2100,720], [700,680]
], dtype=np.int32)

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None, names=COLUMNS
)

gt = gt[(gt.frame >= 2000) & (gt.frame < 2300)]
gt_groups = dict(tuple(gt.groupby("frame")))

files = {
    "Current_best": ROOT / "higher_new_track_thresh.txt",
    "Combined": ROOT / "combined_045_090.txt"
}

mh = mm.metrics.create()
summaries = []

for name, path in files.items():
    pred = pd.read_csv(path, header=None, names=COLUMNS)

    mask = [
        cv2.pointPolygonTest(
            ROI,
            (float(r.x + r.w/2), float(r.y + r.h)),
            False
        ) >= 0
        for r in pred.itertuples(index=False)
    ]

    pred = pred.loc[mask].copy()
    pred_groups = dict(tuple(pred.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame_idx in range(2000, 2300):
        gf = gt_groups.get(frame_idx, gt.iloc[0:0])
        pf = pred_groups.get(frame_idx, pred.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x","y","w","h"]].to_numpy(dtype=float),
            pf[["x","y","w","h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf.id.tolist(),
            pf.id.tolist(),
            distances
        )

    summaries.append(mh.compute(
        acc,
        metrics=[
            "mota", "idf1", "num_switches",
            "precision", "recall",
            "num_false_positives", "num_misses"
        ],
        name=name
    ))

comparison = pd.concat(summaries)

display(mm.io.render_summary(
    comparison,
    formatters=mh.formatters,
    namemap=mm.io.motchallenge_metric_names
))

comparison.to_csv(ROOT / "combined_comparison.csv")


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import pandas as pd
import torch

MODEL_PATH = (
    "/content/football_outputs/training/"
    "yolo26n_football_v1/weights/best.pt"
)
VIDEO_PATH = "/content/football_dataset/mot/clips/117093.mp4"

OUTPUT_DIR = Path("/content/football_outputs/botsort_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START, END = 2000, 2300

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, START)

rows = []
processed = 0

for frame_idx in range(START, END):
    ok, frame = cap.read()
    if not ok:
        break

    processed += 1

    result = model.track(
        frame,
        tracker="botsort.yaml",
        conf=0.35,
        imgsz=1536,
        classes=[0],
        device=0 if torch.cuda.is_available() else "cpu",
        persist=True,
        verbose=False
    )[0]

    boxes = result.boxes

    if boxes is None or boxes.id is None:
        continue

    for box, track_id, score in zip(
        boxes.xyxy.cpu().numpy(),
        boxes.id.int().cpu().numpy(),
        boxes.conf.cpu().numpy()
    ):
        x1, y1, x2, y2 = map(float, box)

        rows.append([
            frame_idx, int(track_id),
            x1, y1, x2-x1, y2-y1,
            float(score), -1, -1, -1
        ])

cap.release()

assert processed == END - START

df = pd.DataFrame(rows, columns=columns)

path = OUTPUT_DIR / "segment_B_botsort_default.txt"
df.to_csv(path, index=False, header=False)

print("Frames processed:", processed)
print("Tracking records:", len(df))
print("Unique IDs:", df.id.nunique())
print("Saved:", path)


In [ ]:

import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

columns = [
    "frame", "id", "x", "y", "w", "h",
    "confidence", "c8", "c9", "c10"
]

roi = np.array([
    [150,570], [350,440], [700,330],
    [1250,175], [1900,115], [2700,130],
    [3400,230], [3950,550],
    [3550,720], [2100,720], [700,680]
], dtype=np.int32)

files = {
    "ByteTrack_Optimized": (
        "/content/football_outputs/tracker_tuning/"
        "higher_new_track_thresh.txt"
    ),
    "BoTSORT_Default": (
        "/content/football_outputs/botsort_comparison/"
        "segment_B_botsort_default.txt"
    )
}

gt = pd.read_csv(
    "/content/football_dataset/mot/117093.txt",
    header=None, names=columns
)
gt = gt[(gt.frame >= 2000) & (gt.frame < 2300)]
gt_groups = dict(tuple(gt.groupby("frame")))

mh = mm.metrics.create()
results = []

for name, file_path in files.items():
    pred = pd.read_csv(file_path, header=None, names=columns)

    mask = [
        cv2.pointPolygonTest(
            roi,
            (float(r.x + r.w/2), float(r.y + r.h)),
            False
        ) >= 0
        for r in pred.itertuples(index=False)
    ]

    pred = pred.loc[mask].copy()
    pred_groups = dict(tuple(pred.groupby("frame")))

    acc = mm.MOTAccumulator(auto_id=True)

    for frame in range(2000, 2300):
        gf = gt_groups.get(frame, gt.iloc[0:0])
        pf = pred_groups.get(frame, pred.iloc[0:0])

        distances = mm.distances.iou_matrix(
            gf[["x","y","w","h"]].to_numpy(dtype=float),
            pf[["x","y","w","h"]].to_numpy(dtype=float),
            max_iou=0.5
        )

        acc.update(
            gf.id.tolist(),
            pf.id.tolist(),
            distances
        )

    results.append(mh.compute(
        acc,
        metrics=[
            "mota", "idf1", "num_switches",
            "precision", "recall",
            "num_false_positives", "num_misses"
        ],
        name=name
    ))

comparison = pd.concat(results)

display(mm.io.render_summary(
    comparison,
    formatters=mh.formatters,
    namemap=mm.io.motchallenge_metric_names
))

comparison.to_csv(
    "/content/football_outputs/botsort_comparison/"
    "segment_B_comparison.csv"
)


In [ ]:

from pathlib import Path
import shutil
import json

DEST = Path(
    "/content/drive/MyDrive/Football_CV/"
    "player_tracking_v1"
)
DEST.mkdir(parents=True, exist_ok=True)

FILES = {
    "model_best.pt": (
        "/content/football_outputs/training/"
        "yolo26n_football_v1/weights/best.pt"
    ),
    "bytetrack_optimized.yaml": (
        "/content/football_outputs/tracker_configs/"
        "higher_new_track_thresh.yaml"
    ),
    "tracking_metrics.csv": (
        "/content/football_outputs/tracker_tuning/"
        "all_segments_newtrack045.csv"
    ),
    "segment_A.txt": (
        "/content/football_outputs/tracker_tuning/"
        "segment_A_newtrack045.txt"
    ),
    "segment_B.txt": (
        "/content/football_outputs/tracker_tuning/"
        "higher_new_track_thresh.txt"
    ),
    "segment_C.txt": (
        "/content/football_outputs/tracker_tuning/"
        "segment_C_newtrack045.txt"
    )
}

for name, src in FILES.items():
    source = Path(src)
    if not source.exists():
        raise FileNotFoundError(source)

    shutil.copy2(source, DEST / name)
    print("Saved:", name)

metadata = {
    "detector": "YOLO26n fine-tuned V1",
    "training_match": "117092",
    "development_validation_match": "117093",
    "tracker": "ByteTrack",
    "imgsz": 1536,
    "confidence": 0.35,
    "new_track_thresh": 0.45,
    "match_thresh": 0.80,
    "roi": "manual pitch ROI V2",
    "evaluation_segments": [
        [0, 300],
        [2000, 2300],
        [4000, 4300]
    ]
}

(DEST / "metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("\nPlayer tracking milestone backed up!")
